# Experiment 3.2: completion-focused modular fine-tuning

Experiment 3.2 inherits the final Experiment 3.1 policy and changes only three behaviors: novelty weight `1.0 -> 2.0`, one-time coverage milestone bonuses, and successful termination at complete map coverage. The CNN, tensor representation, frontier shaping, penalties, deadlocks, action masking, and episode-based epsilon schedule remain unchanged.

The core algorithm now lives in reusable `utils/minigrid_exploration_*` modules. This notebook only defines configuration, runs smoke checks, launches the trainer, and inspects results.

## Modular architecture

- `minigrid_experiment_config.py`: nested experiment/environment/network/training/validation/output configuration.
- `minigrid_exploration_rewards.py` and `minigrid_coverage_rewards.py`: decomposed reward and episode-local milestones/completion.
- `minigrid_deadlocks.py` and `minigrid_exploration_policy.py`: reusable termination tracking and configurable masked action selection.
- `minigrid_exploration_episode.py`: one shared stepping loop for training and evaluation.
- `minigrid_exploration_metrics.py`, `minigrid_exploration_diagnostics.py`, and `minigrid_exploration_checkpoints.py`: structured results and output lifecycle.
- `minigrid_exploration_trainer.py` and `minigrid_exploration_evaluation.py`: configuration-driven orchestration.

In [1]:
from dataclasses import replace
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import torch

from utils import alert_training_complete
from utils.minigrid_exploration import (
    CheckpointManager,
    ContinuationConfig,
    CoverageMilestoneTracker,
    CoverageRewardConfig,
    EnvironmentConfig,
    MiniGridExplorationEvaluator,
    MiniGridExplorationExperimentConfig,
    MiniGridExplorationTrainer,
    OutputConfig,
    episode_linear_epsilon,
    save_comparison_artifacts,
    verify_literal_coverage_attainable,
)
from utils.minigrid_exploration_diagnostics import save_diagnostic_episode
from utils.minigrid_exploration_episode import ExplorationEpisodeRunner

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

experiment_3_model_path = Path("experiment_3/final_model.pth")
experiment_3_1_v2_model_path = Path("experiment_3_1_v2/latest_checkpoint.pth")
experiment_3_2_model_path = Path("experiment_3_2/final_model.pth")

RUN_COMPLETION_PROOF = True
RUN_SMOKE_TESTS = True
RUN_EXPERIMENT_3_2 = True
RUN_FINAL_EVALUATION = True
RUN_MATCHED_COMPARISON = True

CONTINUE_TRAINING = True
CONTINUE_FROM = "best"  # "latest", "best", or an explicit checkpoint path
additional_training_episodes = 1_000
keep_epsilon_at_floor_when_continuing = True

if not experiment_3_1_v2_model_path.exists():
    raise FileNotFoundError(experiment_3_1_v2_model_path.resolve())

Device: cuda
GPU: NVIDIA GeForce RTX 2050


## Experiment configuration

The complete Experiment 3.1 configuration is reconstructed from its checkpoint. Experiment 3.2 is then expressed only as configuration differences via `dataclasses.replace`. This stage uses `CONTINUE_FROM="best"` and `mode="fine_tune"`: policy/history/best metadata continue cumulatively, while replay, optimizer, and target learning state restart for the changed oscillation reward and learning rate.

In [3]:
experiment_3_1_v2_config = MiniGridExplorationExperimentConfig.from_checkpoint(
    name="experiment_3_1__v2_modular_reference",
    checkpoint_path=experiment_3_1_v2_model_path,
    output_directory="experiment_3_1_v2",
)

experiment_3_2_config = replace(
    experiment_3_1_v2_config,
    name="experiment_3_2",
    reward=replace(
        experiment_3_1_v2_config.reward,
        novelty_beta=2.0,
        escalate_oscillation_penalty=True,
        oscillation_penalty_max_multiplier=4,
    ),
    coverage=CoverageRewardConfig(
        milestones={0.75: 5.0, 0.90: 10.0, 0.95: 20.0, 1.00: 50.0},
        completion_threshold=1.0,
        terminate_on_completion=True,
    ),
    training=replace(
        experiment_3_1_v2_config.training,
        num_episodes=1_000, learning_rate=5e-5, epsilon_end=0.02, random_seed=46,
    ),
    validation=replace(
        experiment_3_1_v2_config.validation,
        frequency=100,
        large_validation_frequency=500,
        large_validation_episode_count=100,
        show_discovered_map_in_gif=True,
    ),
    continuation=ContinuationConfig(
        enabled=CONTINUE_TRAINING,
        checkpoint=CONTINUE_FROM,
        additional_episodes=additional_training_episodes,
        keep_epsilon_at_floor=keep_epsilon_at_floor_when_continuing,
        mode="fine_tune",
    ),
    output=OutputConfig(),  # Inferred from name="experiment_3_2".
)

print("Source:", experiment_3_2_config.initial_checkpoint.resolve())
print("Output:", experiment_3_2_config.output.directory.resolve())
print("Novelty beta:", experiment_3_2_config.reward.novelty_beta)
print("Learning rate:", experiment_3_2_config.training.learning_rate)
print("Oscillation penalties:", [
    experiment_3_2_config.reward.oscillation_penalty * multiplier
    for multiplier in range(1, experiment_3_2_config.reward.oscillation_penalty_max_multiplier + 1)
])
print("Milestones:", experiment_3_2_config.coverage.milestone_map)
print("Discovered-map GIF panel:", experiment_3_2_config.validation.show_discovered_map_in_gif)
print("Validation episodes:", len(experiment_3_2_config.validation.checkpoint_seeds))
print("Large validation episodes:", len(experiment_3_2_config.validation.large_validation_seeds))
print("Continue training:", experiment_3_2_config.continuation.enabled)
if experiment_3_2_config.continuation.enabled:
    print("Continue from:", experiment_3_2_config.continuation.checkpoint)
    print("Continuation mode:", experiment_3_2_config.continuation.mode)
    print("Additional episodes:", experiment_3_2_config.continuation.additional_episodes)
print("Full training disabled:", not RUN_EXPERIMENT_3_2)

Source: C:\Users\Lenovo\Documents\GitHub\Mobile-Robotics\experiment_3_1_v2\latest_checkpoint.pth
Output: C:\Users\Lenovo\Documents\GitHub\Mobile-Robotics\experiment_3_2
Novelty beta: 2.0
Learning rate: 5e-05
Oscillation penalties: [-2.0, -4.0, -6.0, -8.0]
Milestones: {0.75: 5.0, 0.9: 10.0, 0.95: 20.0, 1.0: 50.0}
Discovered-map GIF panel: True
Validation episodes: 20
Large validation episodes: 100
Continue training: True
Continue from: best
Continuation mode: fine_tune
Additional episodes: 1000
Full training disabled: False


## Completion attainability proof

A deterministic DFS oracle was run before defining success. Ground-truth geometry was used only to choose a route; `PersistentMiniGridMapper` received ordinary partial observations. Results: both tested 19x19 seeds mapped `361/361` cells, and both tested 27x27 seeds mapped `729/729` cells. Literal canonical coverage `len(mapper.cells)/(width*height) == 1.0` is therefore attainable, so no alternative completion denominator is needed. The proof can be rerun below.

In [4]:
if RUN_COMPLETION_PROOF:
    proof_19 = verify_literal_coverage_attainable(
        experiment_3_2_config.environment, seeds=(0, 20_000)
    )
    environment_27 = replace(
        experiment_3_2_config.environment, width=27, height=27
    )
    proof_27 = verify_literal_coverage_attainable(environment_27, seeds=(0, 20_000))
    completion_reports = proof_19 + proof_27
    assert all(report["literal_completion_attained"] for report in completion_reports)
    print(completion_reports)
else:
    print("Completion proof already passed for two 19x19 and two 27x27 maps.")

[{'seed': 0, 'width': 19, 'height': 19, 'visited_traversable_cells': 259, 'mapped_cells': 361, 'total_map_cells': 361, 'coverage': 1.0, 'steps': 646, 'literal_completion_attained': True}, {'seed': 20000, 'width': 19, 'height': 19, 'visited_traversable_cells': 259, 'mapped_cells': 361, 'total_map_cells': 361, 'coverage': 1.0, 'steps': 645, 'literal_completion_attained': True}, {'seed': 0, 'width': 27, 'height': 27, 'visited_traversable_cells': 562, 'mapped_cells': 729, 'total_map_cells': 729, 'coverage': 1.0, 'steps': 1360, 'literal_completion_attained': True}, {'seed': 20000, 'width': 27, 'height': 27, 'visited_traversable_cells': 567, 'mapped_cells': 729, 'total_map_cells': 729, 'coverage': 1.0, 'steps': 1385, 'literal_completion_attained': True}]


## Focused smoke tests

In [5]:
if RUN_SMOKE_TESTS:
    source_checkpoint = torch.load(
        experiment_3_1_v2_model_path, map_location="cpu", weights_only=False
    )
    assert experiment_3_2_config.network.to_kwargs() == source_checkpoint["architecture"]
    assert experiment_3_2_config.reward.novelty_beta == 2.0
    assert experiment_3_2_config.reward.frontier_progress_beta == source_checkpoint["reward_config"]["frontier_progress_beta"]
    assert experiment_3_2_config.reward.stationary_penalty == source_checkpoint["reward_config"]["stationary_penalty"]
    assert experiment_3_2_config.reward.oscillation_penalty == source_checkpoint["reward_config"]["oscillation_penalty"]
    assert experiment_3_2_config.reward.escalate_oscillation_penalty
    assert experiment_3_2_config.reward.oscillation_penalty_max_multiplier == 4
    assert experiment_3_2_config.reward.blocked_penalty == source_checkpoint["reward_config"]["blocked_penalty"]
    assert experiment_3_2_config.reward.deadlock_penalty == source_checkpoint["reward_config"].get("deadlock_penalty", -1.0)
    assert experiment_3_2_config.training.learning_rate == 5e-5
    assert experiment_3_2_config.training.epsilon_end == 0.02
    assert experiment_3_2_config.deadlock.no_progress_timeout(19, 19) == 76
    assert episode_linear_epsilon(0, 1001, 0.10, 0.02) == 0.10
    assert np.isclose(episode_linear_epsilon(500, 1001, 0.10, 0.02), 0.06)
    assert episode_linear_epsilon(1000, 1001, 0.10, 0.02) == 0.02

    tracker = CoverageMilestoneTracker(experiment_3_2_config.coverage)
    assert tracker.update(0.74, 0.76).bonus == 5.0
    assert tracker.update(0.76, 0.80).bonus == 0.0
    update = tracker.update(0.89, 1.0)
    assert update.crossed_milestones == (0.90, 0.95, 1.0) and update.bonus == 80.0
    assert tracker.update(1.0, 1.0).bonus == 0.0
    tracker.reset()
    assert tracker.update(0.74, 0.76).bonus == 5.0

    trainer = MiniGridExplorationTrainer(experiment_3_2_config, device).initialize()
    assert len(trainer.replay_buffer) == 0
    if experiment_3_2_config.continuation.enabled:
        resume_checkpoint = torch.load(
            trainer.source_checkpoint_path, map_location="cpu", weights_only=False
        )
        assert trainer.completed_episodes == resume_checkpoint["training_episode"]
        assert trainer.total_steps == resume_checkpoint["total_steps"]
        assert len(trainer.training_history) == len(resume_checkpoint["training_metrics"])
        assert trainer.source_checkpoint_path.name == "best_validation_checkpoint.pth"
        if experiment_3_2_config.continuation.mode == "fine_tune":
            assert len(trainer.optimizer.state) == 0
            expected_target = resume_checkpoint["model_state_dict"]
        else:
            expected_optimizer_size = len(
                resume_checkpoint.get("optimizer_state_dict", {}).get("state", {})
            )
            assert len(trainer.optimizer.state) == expected_optimizer_size
            expected_target = resume_checkpoint.get(
                "target_state_dict", resume_checkpoint["model_state_dict"]
            )
        if experiment_3_2_config.continuation.keep_epsilon_at_floor:
            assert trainer._epsilon_for_run_episode(0, additional_training_episodes) == experiment_3_2_config.training.epsilon_end
        for name, resumed in resume_checkpoint["model_state_dict"].items():
            assert torch.equal(trainer.policy_net.state_dict()[name].cpu(), resumed.cpu())
        for name, resumed_target in expected_target.items():
            assert torch.equal(trainer.target_net.state_dict()[name].cpu(), resumed_target.cpu())
    else:
        assert len(trainer.optimizer.state) == 0
        for name, inherited in source_checkpoint["model_state_dict"].items():
            assert torch.equal(trainer.policy_net.state_dict()[name].cpu(), inherited.cpu())
        for policy_value, target_value in zip(
            trainer.policy_net.state_dict().values(), trainer.target_net.state_dict().values()
        ):
            assert torch.equal(policy_value, target_value)
    print("Configuration, milestones, initialization mode, replay, optimizer, target, and epsilon checks passed.")

Configuration, milestones, initialization mode, replay, optimizer, target, and epsilon checks passed.


In [6]:
if RUN_SMOKE_TESTS:
    # A deliberately low test threshold exercises real success termination and logs.
    success_smoke_config = replace(
        experiment_3_2_config,
        environment=replace(experiment_3_2_config.environment, max_steps=20),
        coverage=CoverageRewardConfig(
            milestones={0.12: 7.0}, completion_threshold=0.12, terminate_on_completion=True
        ),
        continuation=ContinuationConfig(),
    )
    terminal_flags = []
    with TemporaryDirectory() as temporary_directory:
        result = ExplorationEpisodeRunner(success_smoke_config, device).run(
            seed=20_000,
            policy_net=trainer.policy_net,
            capture_diagnostics=True,
            transition_handler=lambda state, action, reward, next_state, done: terminal_flags.append(done),
        )
        save_diagnostic_episode(result, temporary_directory)
        assert result.metrics["episode_end_reason"] == "coverage_success"
        assert result.metrics["coverage_milestone_reward"] == 7.0
        assert terminal_flags[-1] is True
        assert any("SUCCESS" in event for event in result.action_log[-1]["milestone_events"])
        assert (Path(temporary_directory) / "seed_20000.gif").exists()

    environment_27 = replace(
        experiment_3_2_config.environment, width=27, height=27, max_steps=4
    )
    large_results, large_summary = trainer.evaluate(
        seeds=(30_000,), environment_override=environment_27
    )
    assert large_summary["random_action_count"] == 0
    assert large_results[0].metrics["no_progress_timeout"] == 2 * (27 + 27)
    print("Coverage-success terminal/GIF and 27x27 greedy checks passed on", device)

Coverage-success terminal/GIF and 27x27 greedy checks passed on cuda


In [7]:
if RUN_SMOKE_TESTS:
    # Exercise the complete trainer/checkpoint path without writing experiment outputs.
    with TemporaryDirectory() as temporary_directory:
        one_episode_config = replace(
            experiment_3_2_config,
            name="experiment_3_2_smoke",
            environment=replace(experiment_3_2_config.environment, max_steps=4),
            training=replace(
                experiment_3_2_config.training, num_episodes=1, batch_size=1,
                minimum_replay_size=1, replay_capacity=32, target_update_frequency=2,
            ),
            validation=replace(
                experiment_3_2_config.validation, seeds=(20_000,),
                checkpoint_seed_count=1, diagnostic_seeds=(20_000,), frequency=1,
                large_validation_frequency=1, large_validation_episode_count=1,
            ),
            continuation=ContinuationConfig(),
            output=OutputConfig(temporary_directory),
        )
        smoke_trainer = MiniGridExplorationTrainer(one_episode_config, device)
        _, smoke_metadata = smoke_trainer.train()
        assert len(smoke_trainer.replay_buffer) > 0
        assert len(smoke_trainer.optimizer.state) > 0
        assert (Path(temporary_directory) / "final_model.pth").exists()
        assert (Path(temporary_directory) / "checkpoints/latest_checkpoint.pth").exists()
        assert (Path(temporary_directory) / "checkpoints/best_validation_checkpoint.pth").exists()
        assert smoke_metadata["reward_config"]["novelty_beta"] == 2.0
    dummy_experiment = replace(
        experiment_3_2_config, name="future_dummy",
        continuation=ContinuationConfig(), output=OutputConfig("future_dummy")
    )
    assert dummy_experiment.name == "future_dummy"
    print("One-episode CUDA trainer and configuration-reuse checks passed.")

Validation 1: coverage=24.4% | median=24.4% | success=0.0% | osc=0.0% | stationary=0.0% | no-progress=0.0% | 50%=0.0% | 75%=0.0% | 90%=0.0% | 95%=0.0% | 100%=0.0%
Large validation 1: episodes=1 | coverage=24.4% | median=24.4% | success=0.0% | osc=0.0% | stationary=0.0% | no-progress=0.0% | 50%=0.0% | 75%=0.0% | 90%=0.0% | 95%=0.0% | 100%=0.0%
One-episode CUDA trainer and configuration-reuse checks passed.


## Experiment 3.2 launch

In [8]:
if RUN_EXPERIMENT_3_2:
    experiment_3_2_trainer = MiniGridExplorationTrainer(
        experiment_3_2_config, device
    )
    experiment_3_2_policy, experiment_3_2_metadata = experiment_3_2_trainer.train()
    alert_training_complete("Experiment 3.2 fine-tuning finished.")
else:
    print("Ready. Set RUN_EXPERIMENT_3_2 = True to start the 1,000-episode run.")

Resuming experiment_3_2
Continuation mode: fine_tune
Previous episodes: 2500
Additional episodes: 1000
Target cumulative episodes: 3500
Replay buffer: fresh (checkpoint replay persistence is unavailable)


c:\Users\Lenovo\Documents\GitHub\Mobile-Robotics\utils\minigrid_exploration_trainer.py:489: RuntimeWarning: Mean of empty slice
  f"loss={np.nanmean([m['training_loss'] for m in recent]):.4f}"


Episode 2525/3500 | run=2.4% | epsilon=0.020 | coverage=78.0% | success=4.0% | milestones=16.20 | loss=nan
Episode 2550/3500 | run=4.9% | epsilon=0.020 | coverage=84.4% | success=4.0% | milestones=15.60 | loss=6.3416
Episode 2575/3500 | run=7.4% | epsilon=0.020 | coverage=76.5% | success=4.0% | milestones=16.80 | loss=5.5897
Episode 2600/3500 | run=9.9% | epsilon=0.020 | coverage=80.5% | success=12.0% | milestones=19.80 | loss=5.2447
Validation 2600: coverage=76.7% | median=85.9% | success=5.0% | osc=50.0% | stationary=15.0% | no-progress=0.0% | 50%=80.0% | 75%=65.0% | 90%=45.0% | 95%=25.0% | 100%=5.0%
Episode 2625/3500 | run=12.4% | epsilon=0.020 | coverage=82.5% | success=8.0% | milestones=21.60 | loss=5.1646
Episode 2650/3500 | run=14.9% | epsilon=0.020 | coverage=77.0% | success=12.0% | milestones=21.00 | loss=5.1971
Episode 2675/3500 | run=17.4% | epsilon=0.020 | coverage=80.5% | success=16.0% | milestones=26.60 | loss=5.1063
Episode 2700/3500 | run=19.9% | epsilon=0.020 | coverag

## Final evaluation and matched comparison

In [9]:
if RUN_FINAL_EVALUATION:
    final_policy, _ = CheckpointManager.load_policy(experiment_3_2_model_path, device)
    evaluator = MiniGridExplorationEvaluator(experiment_3_2_config, device)
    final_results = evaluator.evaluate(final_policy)
    final_summary = evaluator.summarize(final_results)
    assert final_summary["random_action_count"] == 0
    evaluator.evaluate(
        final_policy,
        seeds=experiment_3_2_config.validation.diagnostic_seeds,
        artifact_directory=experiment_3_2_config.output.directory / "validation_artifacts" / "final_evaluation",
    )
    print(final_summary)
else:
    print("Final evaluation is disabled until Experiment 3.2 is trained.")

{'mean_final_coverage': 0.8328531855955679, 'median_final_coverage': 0.9168975069252078, 'mean_traversable_coverage': 0.8488021961023187, 'new_cells_per_step': 2.988086600970901, 'zero_information_ratio': 0.6136011952832552, 'stationary_action_ratio': 0.36518097185514137, 'oscillation_ratio': 0.07868477173230758, 'blocked_forward_ratio': 0.01550910931174089, 'position_change_ratio': 0.6348190281448587, 'mean_maximum_consecutive_stationary_steps': 5.88, 'random_action_count': 0, 'greedy_action_count': 11390, 'random_discovery_actions': 0, 'greedy_discovery_actions': 3613, 'random_new_cells': 0, 'greedy_new_cells': 26723, 'random_discovery_probability': 0.0, 'greedy_discovery_probability': 0.31720807726075506, 'fraction_discovery_events_from_random': 0.0, 'fraction_new_cells_from_random': 0.0, 'fraction_new_cells_from_greedy': 1.0, 'random_position_change_probability': 0.0, 'greedy_position_change_probability': 0.0, 'random_forward_actions': 0, 'greedy_forward_actions': 7697, 'random_for

In [10]:
if RUN_MATCHED_COMPARISON:
    evaluator = MiniGridExplorationEvaluator(experiment_3_2_config, device)
    model_paths = {
        "Experiment 3": experiment_3_model_path,
        "Experiment 3.1.2": experiment_3_1_v2_model_path,
        "Experiment 3.2": experiment_3_2_model_path,
    }
    matched_results = {}
    for label, path in model_paths.items():
        policy, _ = CheckpointManager.load_policy(path, device)
        matched_results[label] = evaluator.evaluate(
            policy, seeds=experiment_3_2_config.validation.seeds
        )
    comparison = save_comparison_artifacts(
        experiment_3_2_config.output.directory / "matched_seed_comparison",
        matched_results,
        thresholds=experiment_3_2_config.validation.coverage_thresholds,
    )
    print(comparison)
else:
    print("Matched Experiment 3/3.1.2/3.2 comparison remains disabled.")

{'Experiment 3': {'mean_final_coverage': 0.499639889196676, 'median_final_coverage': 0.4736842105263158, 'mean_traversable_coverage': 0.5184323266118848, 'new_cells_per_step': 2.590954176183912, 'zero_information_ratio': 0.7128058408084175, 'stationary_action_ratio': 0.45312243351908094, 'oscillation_ratio': 0.1950834490423598, 'blocked_forward_ratio': 0.01354911014911015, 'position_change_ratio': 0.546877566480919, 'mean_maximum_consecutive_stationary_steps': 6.01, 'random_action_count': 0, 'greedy_action_count': 10000, 'random_discovery_actions': 0, 'greedy_discovery_actions': 1965, 'random_new_cells': 0, 'greedy_new_cells': 14694, 'random_discovery_probability': 0.0, 'greedy_discovery_probability': 0.1965, 'fraction_discovery_events_from_random': 0.0, 'fraction_new_cells_from_random': 0.0, 'fraction_new_cells_from_greedy': 1.0, 'random_position_change_probability': 0.0, 'greedy_position_change_probability': 0.0, 'random_forward_actions': 0, 'greedy_forward_actions': 7188, 'random_fo